# NASA Data Explorer - quickstart

Drop in any data file (NetCDF, HDF, GRIB, GeoTIFF, CSV, FITS, PDF...) and get an instant report.
Runs locally or in [Google Colab](https://colab.research.google.com/github/malkawi06/nasa-data-explorer/blob/main/examples/quickstart.ipynb).

In [ ]:
# Install (only needed in Colab / a fresh environment)
import importlib.util

if importlib.util.find_spec("nasa_explorer") is None:
    get_ipython().run_line_magic(
        "pip",
        'install -q "nasa-data-explorer[geo,docs,astro,hdf4] @ git+https://github.com/malkawi06/nasa-data-explorer"',
    )

## 1. Make a tiny sample (replace with your NASA file on hackathon day)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

demo = Path("demo")
demo.mkdir(exist_ok=True)
time = pd.date_range("2001-01-01", periods=48, freq="MS")
lat, lon = np.arange(20, 41, 2.5), np.arange(30, 51, 2.5)
rng = np.random.default_rng(0)
t2m = 290 + 0.05 * np.arange(48)[:, None, None] + rng.normal(0, 0.3, (48, lat.size, lon.size))
xr.Dataset(
    {"t2m": (("time", "lat", "lon"), t2m, {"units": "K", "long_name": "2 m air temperature"})},
    coords={"time": time, "lat": lat, "lon": lon},
).to_netcdf(demo / "sample.nc")

n = 200
pd.DataFrame(
    {
        "acq_date": pd.date_range("2024-07-01", periods=n, freq="6h"),
        "latitude": rng.uniform(29, 33, n),
        "longitude": rng.uniform(35, 39, n),
        "frp": rng.gamma(2, 5, n),
    }
).to_csv(demo / "fires.csv", index=False)
sorted(p.name for p in demo.iterdir())

## 2. Explore one file - the report renders inline and is saved to `reports/`

In [ ]:
from nasa_explorer import explore

rep = explore("demo/sample.nc")

Subset by variable, bounding box (W,S,E,N) and dates:

In [ ]:
rep = explore(
    "demo/sample.nc",
    var="t2m",
    bbox=(35, 25, 45, 35),
    start="2002-01-01",
    end="2003-12-31",
    display=False,
)
rep.analysis["trends"][0]["text"]

## 3. A whole folder (archives are unpacked automatically) + JSON summary for reuse

In [ ]:
import json

reports = explore("demo", display=False)
for r in reports:
    print(f"{r.reader:15} {r.file:12} -> {r.html_path}")
json.loads(reports[0].json_path.read_text())["headline"]

## 4. Ask your files

Works without any API key (returns the best matching passages). With Ollama running or `GEMINI_API_KEY` / `GROQ_API_KEY` / `ANTHROPIC_API_KEY` set, it writes an answer that cites files and pages.

In [ ]:
from nasa_explorer.qa import ask

print(ask("where are the fires and how strong is the radiative power?", "reports"))

## 5. Optional: AI summaries

```python
import os
os.environ["GEMINI_API_KEY"] = "..."   # or use Colab secrets / a .env file
explore("paper.pdf", ai=True, lang="ar")  # Arabic report + structured paper summary
```